In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# 13_streaming_ingestion
# Databricks Free Edition Compatible
# ========================================

from pyspark.sql.functions import *

stream_df = (
    spark.readStream
        .format("csv")
        .option("header", True)
        .schema("""
            patient_id STRING,
            heart_rate INT,
            oxygen_level INT,
            event_time TIMESTAMP
        """)
        .load(f"{source_path}/icu_stream")
)

query = (
    stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .trigger(availableNow=True)
        .option(
            "checkpointLocation",
            f"{checkpoint_path}/icu_stream"
        )
        .start(f"{bronze_path}/icu_stream")
)

query.awaitTermination()